# Continue the original frozen experiment
This notebook is for existing results at SHA `3258b1d47baa734fea102d7f8ea38fdd626d8174`. It launches that revision's original pipeline and reads training CSVs to display progress. It never switches revisions or changes scientific code.

Completed and scientifically failed models are skipped. Recovery preserves interrupted attempts and restarts only interrupted training from update 1. The six completed primary models remain evidence. This original revision cannot run one model per cell: the remaining pipeline runs in one process, with visible progress.

Enable a GPU runtime. Run mount, inspect status, then setup. If the old process is still running in any notebook/runtime, let it continue: do not launch this recovery alongside it. A `running` record is not proof that a process is alive; `updates_completed=0` can be stale. The CSV shows the most recent logged update.


In [ ]:
COMMIT_SHA = "3258b1d47baa734fea102d7f8ea38fdd626d8174"
RECOVER_INFRASTRUCTURE = False  # Set True only if the prior training process has stopped/interrupted.
from google.colab import drive
from pathlib import Path
import subprocess, sys, json, csv, re
from datetime import datetime, timezone
drive.mount("/content/drive")
BASE = Path("/content/drive/MyDrive/ryan_pinn_trial")
REPO = BASE / "repository"
assert (REPO / ".git").is_dir(), "Mount the Drive account containing the original repository and results"

def git(*args):
    return subprocess.check_output(["git", *args], cwd=REPO, text=True).strip()
assert git("rev-parse", "HEAD") == COMMIT_SHA, "Keep the original experiment checkout at its recorded SHA"
assert not git("status", "--porcelain"), "Preserve and inspect source edits before continuing"
print("Original frozen experiment:", COMMIT_SHA)


In [ ]:
def live_runner_processes():
    lines = subprocess.check_output(["ps", "-eo", "pid,args"], text=True).splitlines()[1:]
    return [line.strip() for line in lines if re.search(r"(?:^|[/ ])scripts/run_frozen\.py(?: |$)", line)]

def training_snapshot(repo):
    """Read training status and CSV only. Never read predictions or held-out metrics."""
    snapshot = []
    for n in (6, 12):
        for method in ("vanilla", "rff"):
            for seed in (0, 1, 2):
                root = repo / f"results/raw/n{n:02d}/{method}/seed_{seed}"
                attempts = sorted(p for p in root.glob("attempt_*") if p.is_dir())
                if not attempts:
                    snapshot.append((n, method, seed, "not_started", 0, None, None))
                    continue
                attempt = attempts[-1]
                state = {}
                record = attempt / "run.json"
                if record.exists():
                    state = json.loads(record.read_text(encoding="utf-8"))
                update = state.get("updates_completed", 0)
                loss = None
                log = attempt / "training_log.csv"
                if log.exists():
                    with log.open(newline="", encoding="utf-8") as handle:
                        for row in csv.DictReader(handle):
                            value = row.get("update", "")
                            if value and value.isdigit() and int(value) >= update:
                                update = int(value)
                                loss = row.get("total_loss")
                snapshot.append((n, method, seed, state.get("status", "infrastructure_interrupted"),
                                 update, loss, str(attempt.relative_to(repo))))
    return snapshot

processes = live_runner_processes()
print("Runner processes visible in THIS runtime:", processes or "none")
print("Other Colab runtimes cannot be checked here. Ensure none is still training these results.")
for row in training_snapshot(REPO):
    n, method, seed, status, update, loss, attempt = row
    print(f"n={n} {method} seed={seed}: {status}, last logged update={update}/8000, attempt={attempt}")
# Validate revision metadata without reading held-out results.
for path in (REPO / "results/raw").glob("n*/**/attempt_*/run.json"):
    state = json.loads(path.read_text(encoding="utf-8"))
    recorded_sha = state.get("provenance", {}).get("git_sha")
    assert recorded_sha in (None, COMMIT_SHA), f"Mixed revision record: {path} ({recorded_sha})"
if any(row[3] in ("running", "infrastructure_interrupted") for row in training_snapshot(REPO)):
    print("If the previous process has stopped, set RECOVER_INFRASTRUCTURE = True before the resume cell.")


## Validate the current GPU environment
This uses the requirements and tests from the original revision. A small unrelated derivative check validates CUDA execution; it does not train a protected model. The original per-attempt environment records remain intact.


In [ ]:
assert not live_runner_processes(), "A protected runner is still active in this runtime; wait for it"
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], cwd=REPO, check=True)
cuda_check = """
import torch
assert torch.cuda.is_available(), 'Select a GPU runtime'
print('PyTorch:', torch.__version__, 'CUDA:', torch.version.cuda, 'GPU:', torch.cuda.get_device_name(0))
x = torch.tensor([0.25], device='cuda', requires_grad=True)
dy = torch.autograd.grad(torch.sin(x).sum(), x, create_graph=True)[0]
d2y = torch.autograd.grad(dy.sum(), x)[0]
assert torch.isfinite(d2y).all().item()
torch.cuda.synchronize()
print('CUDA derivative preflight passed')
"""
subprocess.run([sys.executable, "-c", cuda_check], cwd=REPO, check=True)
subprocess.run([sys.executable, "scripts/validate_protocol.py"], cwd=REPO, check=True)
subprocess.run([sys.executable, "-m", "pytest", "-q"], cwd=REPO, check=True)
(BASE / "code_sha.txt").write_text(COMMIT_SHA + "\n", encoding="utf-8")
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
with (BASE / f"environment_resume_{stamp}.txt").open("x", encoding="utf-8") as output:
    subprocess.run([sys.executable, "-m", "pip", "freeze"], stdout=output, check=True)
print("Validation passed. Original SHA retained.")


## Resume with live training progress
Only run this after the old process has stopped. For an interrupted attempt, set `RECOVER_INFRASTRUCTURE = True` above (or in a new cell) before executing this cell. Recovery retains the old attempt and starts a new attempt from update 1. Completed/scientifically failed slots are skipped. The monitor reads logs every 10 seconds and prints changed training statuses and update counts. It never evaluates held-out data. The original runner performs condition evaluation and final deliverable generation under its original gates.


In [ ]:
import threading, signal, time, shutil
assert git("rev-parse", "HEAD") == COMMIT_SHA
assert not git("status", "--porcelain"), "Checkout must remain clean"
assert not live_runner_processes(), "A runner is already active: do not launch another"
interrupted = [row for row in training_snapshot(REPO) if row[3] in ("running", "infrastructure_interrupted")]
assert not interrupted or RECOVER_INFRASTRUCTURE, (
    "Interrupted/running records exist. After confirming the old process has stopped, "
    "set RECOVER_INFRASTRUCTURE = True, then rerun this cell."
)
PIPELINE_COMPLETED = False
command = [sys.executable, "-u", "scripts/run_frozen.py", "--config", "configs/frozen.yaml"]
if RECOVER_INFRASTRUCTURE:
    command.append("--recover-infrastructure")
log_path = BASE / ("console_resume_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ") + ".log")
if not (BASE / "EXPERIMENT_LOG_execution.md").exists():
    shutil.copy2(REPO / "EXPERIMENT_LOG.md", BASE / "EXPERIMENT_LOG_execution.md")
with (BASE / "EXPERIMENT_LOG_execution.md").open("a", encoding="utf-8") as journal:
    journal.write(f"\n{datetime.now(timezone.utc).isoformat()}: Launch original SHA {COMMIT_SHA}; "
                  f"explicit infrastructure recovery={RECOVER_INFRASTRUCTURE}; console={log_path}.\n")
print("Original command:", " ".join(command))
print("Console and progress log:", log_path)
stop = threading.Event()
lock = threading.Lock()
with log_path.open("x", encoding="utf-8") as output:
    def emit(text):
        with lock:
            print(text, end="", flush=True)
            output.write(text)
            output.flush()
    def watch_progress():
        seen = {}
        while not stop.is_set():
            try:
                for row in training_snapshot(REPO):
                    n, method, seed, status, update, loss, attempt = row
                    key = (n, method, seed)
                    if seen.get(key) != row:
                        emit(f"[progress] n={n} {method} seed={seed}: {status} "
                             f"{update}/8000 ({100*update/8000:.1f}%) loss={loss} attempt={attempt}\n")
                        seen[key] = row
            except (OSError, ValueError, KeyError, TypeError) as error:
                emit(f"[monitor] Transient read error; retrying: {error}\n")
            stop.wait(10)
    process = subprocess.Popen(command, cwd=REPO, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
    monitor = threading.Thread(target=watch_progress, daemon=True)
    monitor.start()
    try:
        for line in process.stdout:
            emit(line)
        returncode = process.wait()
    except BaseException:
        if process.poll() is None:
            process.send_signal(signal.SIGINT)
            try:
                process.wait(timeout=20)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
        raise
    finally:
        stop.set()
        monitor.join(timeout=15)
if returncode:
    raise subprocess.CalledProcessError(returncode, command)
PIPELINE_COMPLETED = True
print("Original pipeline finished; deliverables generated. Run the packaging cell next.")


## Package all evidence
The ZIP contains original source, frozen assets, all attempts, per-attempt provenance, processed tables, figures, report, console/progress logs, environment records, and the external journal. Generated files stay in Drive. Do not run the new revision's training cells on this experiment.


In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files
assert PIPELINE_COMPLETED, "Resume cell must finish successfully before packaging"
archive = BASE / ("ryan_pinn_original_submission_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ") + ".zip")
excluded = {".git", "__pycache__", ".pytest_cache", "smoke_outputs"}
with ZipFile(archive, "x", ZIP_DEFLATED) as z:
    for path in BASE.rglob("*"):
        if path.is_file() and path.suffix != ".zip" and not excluded.intersection(path.relative_to(BASE).parts):
            z.write(path, path.relative_to(BASE))
print("Saved:", archive)
files.download(str(archive))
